In [1]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

documents = load_faq_data()
index = build_index(documents)


100%|██████████| 1208/1208 [00:05<00:00, 237.64it/s]


In [2]:
index

<Elasticsearch(['http://localhost:9200'])>

In [9]:
from openai import OpenAI

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    es_client=index,                 # your Elasticsearch client object
    index_name="faq_documents",                # <-- replace with your real ES index name
    llm_client=OpenAI(),
    course="data-engineering-zoomcamp", #"mlops-zoomcamp",
    top_k =10,
    instructions=instructions,
    # top_k=10
)

answer = assistant.rag("How do I run kafka?")
print(answer)

If you mean running the Kafka-related code from the course, the FAQ gives these examples:

- **Java Kafka producer/consumer/kstreams in terminal:**
```bash
java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java
```

- **Running Kafka CLI tools via Docker** is also shown, for example:
```bash
docker run --rm -it --network pyflink_default confluentinc/cp-kafka:7.6.0 kafka-console-consumer --bootstrap-server redpanda:29092 --topic rides --from-beginning
```

If you meant something else by “run kafka,” tell me whether you’re using **Java**, **Python**, or **Docker**, and I’ll point you to the matching command.


In [6]:
print(index.indices.exists(index="faq_documents"))
print(index.count(index="faq_documents")["count"])

True
1208


## Try function injection to replace the search function

In [10]:
from rag_helper_function_injection import RAGBase

In [14]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [20]:
from openai import OpenAI

# Standalone injected function (NO `self` argument)
def search(query, course=None, top_k=5, **kwargs):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": course or "data-engineering-zoomcamp"}

    return index.search(
        query,
        num_results=top_k,   # map RAGBase top_k -> minsearch num_results
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

assistant = RAGBase(
    es_client=index,                    # not used by custom search, but fine to keep
    index_name="faq_documents",
    llm_client=OpenAI(),
    course="data-engineering-zoomcamp",
    instructions=instructions,
    search_fn=search                    # inject standalone function
)

answer = assistant.rag("How do I run kafka?")
print(answer)

To run Kafka-related Python code locally, create and activate a virtual environment, then install the required packages:

```bash
python -m venv env
```

Activate it:

- macOS/Linux:
  ```bash
  source env/bin/activate
  ```
- Windows:
  ```bash
  env\Scripts\activate
  ```

Then install dependencies:

```bash
pip install -r ../requirements.txt
```

When you’re done:

```bash
deactivate
```

Also, make sure the Docker images are running before executing the Python file.
